<a href="https://colab.research.google.com/github/saruko/drifting-hypernova/blob/main/%E3%83%9E%E3%82%A6%E3%82%B9%E3%81%B2%E3%81%A3%E3%81%8B%E3%81%8D%E8%A1%8C%E5%8B%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# セル1: 環境構築
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import h5py
from scipy.signal import butter, filtfilt, hilbert, find_peaks

print("✅ インポート完了")

In [ ]:
# ============================================================
# セル2: SLEAP H5読み込み（前回と同じ・堅牢版）
# ============================================================

def load_sleap_analysis(h5_path, fps=60):
    with h5py.File(h5_path, 'r') as f:
        print("📂 H5 keys:", list(f.keys()))
        tracks = f['tracks'][:]
        node_names = [n.decode() if isinstance(n, bytes) else n for n in f['node_names'][:]]
        track_names = [n.decode() if isinstance(n, bytes) else n for n in f['track_names'][:]]
        n_tracks = len(track_names)

        occupancy = None
        if 'track_occupancy' in f:
            occ = f['track_occupancy'][:]
            if occ.shape[0] == n_tracks and occ.shape[1] == tracks.shape[0]:
                occupancy = occ.T
            elif occ.shape[0] == tracks.shape[0] and occ.shape[1] == n_tracks:
                occupancy = occ
            else:
                occupancy = None

        # shape正規化 → (frames, nodes, xy, animals)
        if tracks.ndim == 4:
            if tracks.shape[0] > tracks.shape[3] and tracks.shape[3] == n_tracks:
                pass
            elif tracks.shape[0] == n_tracks and tracks.shape[3] > tracks.shape[0]:
                tracks = np.transpose(tracks, (3, 1, 2, 0))
            elif tracks.shape[1] == n_tracks:
                tracks = np.transpose(tracks, (0, 2, 3, 1))
        elif tracks.ndim == 3:
            tracks = tracks[:, :, :, np.newaxis]

        print(f"✅ Normalized shape: {tracks.shape}")
        print(f"   Nodes: {node_names}")
        return tracks, node_names, track_names, occupancy, fps

# --- パス設定 ---
H5_PATH = "/content/drive/MyDrive/mouse_project/predictions.analysis.h5"
# H5_PATH = "/content/predictions.analysis.h5"  # テスト用

tracks, node_names, track_names, occupancy, fps = load_sleap_analysis(H5_PATH, fps=60)
n_frames, n_animals = tracks.shape[0], tracks.shape[3]
time_sec = np.arange(n_frames) / fps

In [ ]:
# ============================================================
# セル3: 特徴量計算（速度・距離・欠損値対策）
# ============================================================

required = ['nose', 'left_ear', 'right_ear', 'left_hindpaw', 'right_hindpaw']
available = [n for n in required if n in node_names]
missing = [n for n in required if n not in node_names]
if missing:
    print(f"⚠️ 欠損ノード: {missing} → 類似名を探索")
    for m in missing:
        cands = [n for n in node_names if m.replace('_', '') in n.replace('_', '')]
        if cands:
            print(f"   '{m}' → '{cands[0]}'")
            available.append(cands[0])

node_idx = {n: node_names.index(n) for n in available if n in node_names}

features = []
for animal in range(n_animals):
    valid_frames = 0
    for frame in range(1, n_frames):
        if occupancy is not None and not (occupancy[frame, animal] and occupancy[frame-1, animal]):
            continue

        def get_coord(name, f):
            if name not in node_idx:
                return None
            c = tracks[f, node_idx[name], :, animal]
            if np.isnan(c).any() or (c == 0).all():
                return None
            return c

        nose = get_coord('nose', frame)
        prev_nose = get_coord('nose', frame-1)
        left_hind = get_coord('left_hindpaw', frame)
        prev_left = get_coord('left_hindpaw', frame-1)
        right_hind = get_coord('right_hindpaw', frame)
        prev_right = get_coord('right_hindpaw', frame-1)
        left_ear = get_coord('left_ear', frame)
        prev_lear = get_coord('left_ear', frame-1)
        right_ear = get_coord('right_ear', frame)
        prev_rear = get_coord('right_ear', frame-1)

        if any(v is None for v in [nose, left_hind, right_hind, left_ear, right_ear,
                                    prev_nose, prev_left, prev_right, prev_lear, prev_rear]):
            continue

        valid_frames += 1
        left_speed = np.linalg.norm(left_hind - prev_left) * fps
        right_speed = np.linalg.norm(right_hind - prev_right) * fps
        hind_speed = (left_speed + right_speed) / 2
        left_dist = np.linalg.norm(left_hind - nose)
        right_dist = np.linalg.norm(right_hind - nose)
        hind_dist = (left_dist + right_dist) / 2
        head_speed = (np.linalg.norm(left_ear - prev_lear) + np.linalg.norm(right_ear - prev_rear)) / 2 * fps

        features.append({
            'frame': frame, 'animal': animal, 'time_sec': time_sec[frame],
            'hind_speed': hind_speed, 'hind_dist': hind_dist, 'head_speed': head_speed
        })
    print(f"   Animal {animal}: {valid_frames} valid frames")

feature_df = pd.DataFrame(features)

In [ ]:
# ============================================================
# セル4: 【修正版 v2】教師なし自動検出 — 欠損フレーム対応 + 閾値バグ修正
# ============================================================

from scipy.signal import butter, filtfilt, hilbert, find_peaks

# --- 事前キャリブレーション済み固定パラメータ ---
LOW_HZ = 8.0      # バンドパス下限（診断コードで推定した値）
HIGH_HZ = 12.0    # バンドパス上限（診断コードで推定した値）
REACH_PX = 110.0  # 後肢が頭に届く最大距離（診断コードで推定した値）

def bandpass_envelope_safe(signal, fs, low=8, high=12, order=4):
    """連続等間隔信号を前提としたバンドパス + Hilbert変換"""
    nyq = fs / 2
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    filtered = filtfilt(b, a, signal)
    envelope = np.abs(hilbert(filtered))
    return filtered, envelope

# 結果列を初期化
feature_df['scratch_score'] = np.nan
feature_df['bp_filtered'] = np.nan
feature_df['scratch_score_filtered'] = 0.0
feature_df['cycle_peaks'] = 0

print(f"🎯 固定パラメータ: {LOW_HZ}-{HIGH_HZ} Hz, REACH={REACH_PX}px")

for animal in feature_df['animal'].unique():
    # --- 1. 個体データをフレーム順にソート ---
    sub_df = feature_df[feature_df['animal'] == animal].sort_values('frame')
    indices = sub_df.index                    # feature_df 書き戻し用
    frames = sub_df['frame'].values.astype(int)
    speed = sub_df['hind_speed'].values
    dist = sub_df['hind_dist'].values

    if len(frames) < fps * 2:
        print(f"   Animal {animal}: データ不足({len(frames)}フレーム) → スキップ")
        continue

    # --- 2. 連続配列を作成（フレーム番号 → 物理インデックス）---
    full_speed = np.full(n_frames, np.nan, dtype=np.float64)
    full_dist = np.full(n_frames, np.nan, dtype=np.float64)
    full_speed[frames] = speed
    full_dist[frames] = dist

    # --- 3. 線形補間（欠損フレームを埋めて連続化）---
    # np.interp: 範囲外は端点値で定値外挿（extrapolation）
    t_all = np.arange(n_frames)
    valid_speed = ~np.isnan(full_speed)
    valid_dist = ~np.isnan(full_dist)

    if valid_speed.sum() < fps * 2:
        continue

    full_speed = np.interp(t_all, t_all[valid_speed], full_speed[valid_speed])
    full_dist = np.interp(t_all, t_all[valid_dist], full_dist[valid_dist])

    # --- 4. バンドパスフィルタ（連続等間隔信号に適用）---
    filtered, envelope = bandpass_envelope_safe(
        full_speed, fs=fps, low=LOW_HZ, high=HIGH_HZ
    )

    # --- 5. 正規化 ---
    if envelope.max() > 0:
        envelope_norm = envelope / envelope.max()
    else:
        envelope_norm = np.zeros_like(envelope)

    # --- 6. 距離フィルタ（後肢が頭に届かない = 歩行中）---
    score = envelope_norm.copy()
    score[full_dist > REACH_PX] = 0

    # --- 7. 元の有効フレームのみを抽出して書き戻し ---
    feature_df.loc[indices, 'bp_filtered'] = filtered[frames]
    feature_df.loc[indices, 'scratch_score'] = envelope_norm[frames]
    feature_df.loc[indices, 'scratch_score_filtered'] = score[frames]

    # --- 8. 周期ピーク検出（連続信号上で実行）---
    # distance: 上限周波数の周期に相当するフレーム数（偽ピーク抑制）
    min_peak_dist = max(1, int(fps / HIGH_HZ))  # e.g. 60fps / 12Hz = 5 frames
    peaks, _ = find_peaks(filtered, distance=min_peak_dist, prominence=0.01)

    # 【修正】set に変換して O(1) ルックアップ
    frames_set = set(frames)
    valid_peaks = [p for p in peaks if p in frames_set]
    peak_map = {f: 1 for f in valid_peaks}
    feature_df.loc[indices, 'cycle_peaks'] = [peak_map.get(f, 0) for f in frames]

    print(f"   Animal {animal}: 有効フレーム {len(frames)} → 完了")

bg = feature_df.loc[
    feature_df['scratch_score_filtered'] < 0.2,
    'scratch_score_filtered'
].dropna()

if len(bg) > 10:
    th_mean = bg.mean()
    th_std = bg.std()

    # 方法A: mean + 4σ（正規分布近似で背景99.997%を除外）
    th_a = th_mean + 4 * th_std

    # 方法B: 99パーセンタイル（背景分布の上端、経験的な上限）
    th_b = np.percentile(bg, 99.0)

    # 【修正】th_a と th_b の大きい方を採用（より保守的・背景漏れ防止）
    # さらに [0.1, 0.4] にクリップして暴走を防止
    auto_threshold = np.clip(max(th_a, th_b), 0.1, 0.4)

    print(f"   背景統計: mean={th_mean:.4f}, std={th_std:.4f}, 99%ile={th_b:.4f}")
    print(f"   mean+4σ={th_a:.4f}, 99%ile={th_b:.4f} → 閾値={auto_threshold:.4f}")
else:
    auto_threshold = 0.3
    print(f"   背景サンプル不足({len(bg)}件) → デフォルト閾値: {auto_threshold}")

feature_df['predicted_scratch'] = (
    feature_df['scratch_score_filtered'] > auto_threshold
).astype(int)

print(f"\n✅ 自動検出完了: scratchフレーム = {feature_df['predicted_scratch'].sum()}")

In [ ]:
# ============================================================
# セル4.5（デバッグ確認用）
# ============================================================

print("=== 閾値パラメータ確認 ===")
print(f"実際のauto_threshold: {auto_threshold:.4f}")
print(f"th_a (mean+4σ): {th_a:.4f}")
print(f"th_b (99%ile): {th_b:.4f}")
print(f"clip後検証: {np.clip(max(th_a, th_b), 0.1, 0.4):.4f}")

print("\n=== 個体別検出フレーム数 ===")
print(feature_df.groupby('animal')['predicted_scratch'].sum())

print("\n=== 閾値ロジック妥当性チェック ===")
# 閾値がth_aとth_bの両方を上回っているか確認
assert auto_threshold >= min(th_a, th_b), "閾値が両推定値を下回っています！"
assert 0.1 <= auto_threshold <= 0.4, "閾値がクリップ範囲外です！"
print("✅ 全チェック通過")

In [ ]:
# ============================================================
# セル5: イベント抽出（状態遷移型・前回修正版）
# ============================================================

results = []
MIN_DURATION_SEC = 0.15  # 150ms未満はノイズ

for animal in feature_df['animal'].unique():
    adf = feature_df[feature_df['animal'] == animal].sort_values('frame').reset_index(drop=True)

    in_event = False
    start_frame = start_time = None
    cycle_count = 0

    for _, row in adf.iterrows():
        pred = row['predicted_scratch']
        frame = int(row['frame'])
        t = row['time_sec']

        if pred == 1 and not in_event:
            in_event = True
            start_frame = frame
            start_time = t
            cycle_count = int(row['cycle_peaks'])
        elif pred == 1 and in_event:
            cycle_count += int(row['cycle_peaks'])
        elif pred == 0 and in_event:
            in_event = False
            duration = t - start_time
            if duration >= MIN_DURATION_SEC:
                results.append({
                    'animal': animal,
                    'start_frame': start_frame, 'end_frame': frame,
                    'start_time': start_time, 'end_time': t,
                    'duration_sec': duration,
                    'duration_frames': frame - start_frame,
                    'estimated_cycles': cycle_count  # 引っ掻き回数（推定）
                })
            cycle_count = 0

    if in_event:
        last = adf.iloc[-1]
        duration = last['time_sec'] - start_time
        if duration >= MIN_DURATION_SEC:
            results.append({
                'animal': animal,
                'start_frame': start_frame, 'end_frame': int(last['frame']),
                'start_time': start_time, 'end_time': last['time_sec'],
                'duration_sec': duration,
                'duration_frames': int(last['frame']) - start_frame,
                'estimated_cycles': cycle_count
            })

results_df = pd.DataFrame(results)
print(f"✅ イベント抽出: {len(results_df)}件")
print(results_df.head(10))

In [ ]:
# ============================================================
# セル6: 個体別集計
# ============================================================

if len(results_df) > 0:
    summary = results_df.groupby('animal').agg(
        scratch_count=('duration_sec', 'count'),
        total_duration_sec=('duration_sec', 'sum'),
        mean_duration_sec=('duration_sec', 'mean'),
        mean_cycles=('estimated_cycles', 'mean'),
        total_cycles=('estimated_cycles', 'sum')
    )
    total = pd.DataFrame({
        'scratch_count': [results_df['duration_sec'].count()],
        'total_duration_sec': [results_df['duration_sec'].sum()],
        'mean_duration_sec': [results_df['duration_sec'].mean()],
        'mean_cycles': [results_df['estimated_cycles'].mean()],
        'total_cycles': [results_df['estimated_cycles'].sum()]
    }, index=['TOTAL'])
    summary = pd.concat([summary, total])
else:
    summary = pd.DataFrame()

print("✅ 集計完了")
print(summary)

# CSV保存
OUT = "/content/drive/MyDrive/mouse_project/outputs/auto_scratch_summary.csv"
import os
os.makedirs(os.path.dirname(OUT), exist_ok=True)
summary.to_csv(OUT)
print(f"💾 保存: {OUT}")

In [ ]:
# ============================================================
# セル7: 可視化（自動検出結果）
# ============================================================

fig, axes = plt.subplots(n_animals, 1, figsize=(14, 3 * n_animals), sharex=True)
if n_animals == 1:
    axes = [axes]

for idx, animal in enumerate(sorted(feature_df['animal'].unique())):
    ax = axes[idx]
    adf = feature_df[feature_df['animal'] == animal].sort_values('frame')

    # 後肢速度（薄い青）
    ax.plot(adf['time_sec'], adf['hind_speed'], color='lightblue', alpha=0.5, label='Hind Speed')

    # バンドパス包絡線（赤・scratchスコア）
    ax2 = ax.twinx()
    ax2.plot(adf['time_sec'], adf['scratch_score_filtered'], color='red', alpha=0.8, linewidth=1.5, label='Scratch Score')
    ax2.axhline(y=auto_threshold, color='red', linestyle='--', alpha=0.5, label=f'Threshold={auto_threshold:.2f}')
    ax2.set_ylabel('Scratch Score (0-1)', color='red')
    ax2.tick_params(axis='y', labelcolor='red')

    # 検出イベントハイライト
    evs = results_df[results_df['animal'] == animal]
    for _, ev in evs.iterrows():
        ax.axvspan(ev['start_time'], ev['end_time'], color='magenta', alpha=0.15, label='Detected' if _ == evs.index[0] else "")

    ax.set_title(f'Animal {animal} — Unsupervised Scratch Detection')
    ax.set_ylabel('Speed (px/s)')
    ax.legend(loc='upper left')
    ax2.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (seconds)')
plt.suptitle('Automated Scratching Detection (No Manual Labels)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\n🎉 完全自動解析完了（手動ラベル不要）")